# Phase I: how do the three models behave?

Black box only: inputs, outputs, labels. Everything here reads the tables that
`python -m aivis.phase1.run_all` writes to `outputs/phase1/tables/`. Each section has the
question, the hypothesis from the plan (C1 to C11), the view, and a cell for what we see.

Before looking at a view for the first time, write the expectation into `logs/preregistration.md`.

In [ ]:
%matplotlib inline
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from aivis.phase1 import views as V, core, build, predict_all
V.style()
pd.set_option("display.width", 140)
T = lambda name: pd.read_parquet(core.TABLES / name)
predict_all.main(["--status"])

## 0. Checks before anyone analyses

Our matcher must give exactly the official scorer's counts, and the published operating points
should land near the known numbers (YOLO about 0.43 macro F1 on dev in the task paper, NAVE about
0.51 LOSO in our DCASE paper, which used LOSO over all 11 dev site-years, so ours on 3 sites differs).

In [ ]:
from aivis import evaluate
site = "casey2017"
gt = core.load_labels("validation"); g = gt[gt.dataset == site].reset_index(drop=True)
for m in ["yolo", "whalevad", "nave"]:
    ev = core.events_at(m, site, build.thr_for(m, site, "default"))
    ours, _ = core.score(ev, g)
    det = pd.DataFrame({"dataset": ev.dataset, "filename": ev.filename, "annotation": ev.cls,
                        "start_datetime": pd.to_datetime(ev.t0, unit="s").dt.strftime("%Y-%m-%dT%H:%M:%S.%f") + "+00:00",
                        "end_datetime": pd.to_datetime(ev.t1, unit="s").dt.strftime("%Y-%m-%dT%H:%M:%S.%f") + "+00:00"})
    off, _ = evaluate.evaluate(det, evaluate.load_labels("validation", [site]), 0.3, None, strict=True)
    a = ours[ours.dataset == site].set_index("cls")[["tp", "fp", "fn"]].sort_index()
    b = off[off.dataset == site].rename(columns={"class": "cls"}).set_index("cls")[["tp", "fp", "fn"]].sort_index().astype(int)
    print(m, "identical to the official scorer:", a.equals(b))

In [ ]:
r = T("ranking.parquet")
r[r.cls.isna() & (r.iou == 0.3)].pivot_table(index=["thresholds", "dataset"], columns="model", values="macro_B").round(3)

In [ ]:
import json; print(json.dumps(build.thresholds(), indent=1)[:3000])

## 1. Error anatomy (idea 1; C2, C3, C11)

Every label gets an outcome per model (hit, fragmented, merged, near miss, wrong class, missed
almost, missed silent), every detection one too (tp, duplicate, boundary, cross class, near call,
background). Three ways of comparing (Gleicher): side by side, explicit encoding, superposition.

**Juxtaposition**: one outcome bar per model, small multiples per site and class.

In [ ]:
V.fig_label_outcomes(); plt.show()

In [ ]:
V.fig_event_outcomes(); plt.show()

**Explicit encoding**: which models hit each label, and which fire in each false-alarm region.

In [ ]:
V.fig_upsets(); plt.show()

In [ ]:
w = T("labels_wide_loso.parquet")
pd.crosstab([w.dataset, w.cls], w.hit_combo, margins=True)

**Superposition**: the comparison browser. Pick labels by model and outcome, or false-alarm regions by
which models fired. Spectrogram with labels (black) and YOLO boxes (dashed blue), Whale-VAD and NAVE
score curves with their thresholds below, YOLO's raw box confidences at the bottom. Tick "audio x10"
to hear it ten times faster.

Questions: which labels does only one model find, and what do they have in common? Which false
alarms do all three share? Is casey2017 BP bad for all three?

In [ ]:
V.browser()

In [ ]:
# or one place directly: V.show_region("casey2017", "2017-03-07T17-00-00_000.wav", 1500, span=90); plt.show()

In [ ]:
## What we see (idea 1):

## 2. Box style and fine labels (idea 6; C3, C10)

Does YOLO copy the box edges more closely than the frame models? Is D two problems (bmd and bpd)?

In [ ]:
V.fig_box_style(); plt.show()

In [ ]:
V.fig_fine_labels(); plt.show()

In [ ]:
V.fig_recall_vs(); plt.show()

In [ ]:
V.fig_recall_vs("duration", np.arange(0, 21, 2), xlabel="label duration (s)"); plt.show()

In [ ]:
lab = T("labels_wide_loso.parquet")
lab["month"] = lab.month.astype(int)
lab.pivot_table(index=["dataset", "month"], values=[c for c in lab if c.startswith("hit_") and c != "hit_combo"], aggfunc="mean").round(2)

In [ ]:
## What we see (idea 6):

## 3. Thresholds and ranking stability (idea 5; C1, H11)

F1 against the threshold per site, then the ranking under 18 scoring variants (default, in-sample and
LOSO thresholds x IoU 0.1/0.3/0.5 x macro A/B), YOLO also with temporal NMS. If the lines cross a lot,
the dev ranking does not mean much.

In [ ]:
V.fig_threshold_curves(); plt.show()

In [ ]:
V.fig_bump(); plt.show()

In [ ]:
r = T("ranking.parquet")
r[r.cls.notna() & (r.iou == 0.3) & (r.thresholds == "loso")].pivot_table(
    index=["dataset", "cls"], columns="model", values=["precision", "recall", "f1"]).round(2)

In [ ]:
## What we see (idea 5):

## 4. Agreement on the new sites (idea 2; C1, C9, C11)

No labels on the evaluation sites, but three models trained on the same labels. Detections per hour,
the share another model confirms, and a rough precision estimate from how often each combination is
real on the dev sites (it assumes agreement means the same at the new sites, which we cannot take
for granted). The blind audit sample is in `outputs/phase1/audit/audit_sheet.csv` (key in a separate
file, do not open it before labelling).

In [ ]:
V.fig_agreement(); plt.show()

In [ ]:
T("combo_precision_dev.parquet").pivot_table(index="combo", columns="cls", values=["precision_dev", "n_dev"]).round(2)

In [ ]:
for c in ["bmabz", "d", "bp"]:
    V.fig_timeline(c); plt.show()

In [ ]:
st = T("site_stats.parquet")
st.groupby("dataset")[[c for c in st if c.endswith("_db")]].median().round(1)

In [ ]:
## What we see (idea 2):

## 5. Probes (ideas 3 and 4; C2, C4 to C9)

Real dev calls changed in one way at a time; each model runs its normal pipeline. Detection at the
in-sample thresholds (IoU > 0.3 with the call), peak = highest score inside the call. The "base" clip is
the untouched audio; "shift 0" is the cut-out call pasted back into a call-free background of its own
site, the reference for shift, truncation, background swap and SNR.

In [ ]:
pr = V.probe_table()
pr.groupby(["probe", "model"]).detected.mean().unstack().round(2)

**Level (C5)**: if C5 holds, Whale-VAD bends, NAVE bends less, YOLO stays flat.

In [ ]:
V.fig_probe_curves("level"); plt.show(); V.fig_probe_curves("level", y="peak"); plt.show()

**SNR curves (C4, H4)** and SNR50 per background.

In [ ]:
V.fig_probe_curves("snr"); plt.show(); V.snr50_table().round(1)

**Distant transient (C4, C8)**: change in detection vs the unchanged clip.

In [ ]:
V.fig_transient(); plt.show()

**Frequency shift (C6)**, **context (C7)**, **position (C8)**, **truncation (C2)**.

In [ ]:
for p in ["shift", "context", "position"]:
    V.fig_probe_curves(p); plt.show()

In [ ]:
tr = pr[pr.probe.str.startswith("trunc")]
ref = pr[(pr.probe == "shift") & (pr.value == 0)].assign(probe="trunc_start", value=1.0)
pd.concat([tr, ref, ref.assign(probe="trunc_end")]).pivot_table(index=["cls", "probe"], columns=["model", "value"], values="detected").round(2)

**Background swap (C9)** and **density (C7, BP only)**.

In [ ]:
V.fig_bgswap(); plt.show()

In [ ]:
d = pr[pr.cls == "bp"]
d[d.probe.isin(["density"]) | ((d.probe == "shift") & (d.value == 0))].pivot_table(index="probe", columns="model", values=["detected", "peak"]).round(2)

In [ ]:
## What we see (probes):